# Fooocus Easy Colab (run-all, DIVERSE DETAIL preset)

1) Set Colab runtime to GPU.
2) Run all cells.
3) Open the printed link `✅ Open this URL`.

This notebook starts Fooocus in background and gives you a stable Colab proxy URL.
Default preset: `realistic_diverse_detail` (better body/skin detail with low face-lock risk).
If needed, change `preset_name` in launch cell to `realistic_diverse` for cleaner prompt-following.

The launch cell now waits for real HTTP readiness before showing the URL (to avoid early proxy 500).


In [ ]:
!pip install -q pygit2==1.15.1

%cd /content
!if [ ! -d "/content/Fooocus/.git" ]; then git clone -b cursor/git-eaf0 https://github.com/sunsideaspect/foocus_new.git Fooocus; fi
%cd /content/Fooocus
!git fetch origin cursor/git-eaf0
!git checkout cursor/git-eaf0
!git pull origin cursor/git-eaf0

%env TORCH_COMMAND=pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

import torch
print('Torch:', torch.__version__, '| CUDA:', torch.version.cuda, '| cuda_available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU runtime is required. Colab: Runtime -> Change runtime type -> GPU, then rerun.')


In [ ]:
import os
import subprocess
import sys
import time
import urllib.error
import urllib.request

import torch
from IPython.display import HTML, display
from google.colab import output

%cd /content/Fooocus

PORT = 7865
HEALTH_URL = f"http://127.0.0.1:{PORT}/"
MAX_WAIT_SECONDS = 15 * 60
POLL_SECONDS = 2
MAX_RESTARTS = 1

log_path = "/content/fooocus.log"

preset_name = "realistic_diverse_detail"
total_vram_mb = 0
if torch.cuda.is_available():
    total_vram_mb = int(torch.cuda.get_device_properties(0).total_memory / (1024 * 1024))
vram_flag = "--always-high-vram" if total_vram_mb >= 20000 else "--always-normal-vram"

cmd = [
    sys.executable,
    "entry_with_update.py",
    "--preset", preset_name,
    "--listen", "0.0.0.0",
    "--port", str(PORT),
    vram_flag,
    "--disable-in-browser"
]


def try_get_proxy_url(port: int):
    try:
        return output.eval_js(f"google.colab.kernel.proxyPort({port})")
    except Exception:
        return None


def local_http_ready(url: str) -> bool:
    try:
        with urllib.request.urlopen(url, timeout=2) as r:
            return r.status in (200, 301, 302, 307, 308)
    except urllib.error.HTTPError as e:
        # Some servers may return redirects/errors before UI is fully ready.
        return e.code in (301, 302, 307, 308)
    except Exception:
        return False


def read_log_tail(path: str, max_chars: int = 30000) -> str:
    if not os.path.exists(path):
        return ""
    with open(path, "rb") as f:
        f.seek(0, os.SEEK_END)
        size = f.tell()
        f.seek(max(0, size - max_chars), os.SEEK_SET)
        return f.read().decode("utf-8", errors="ignore")


def launch(clear_log: bool):
    subprocess.run(["bash", "-lc", "pkill -f 'entry_with_update.py'"], check=False)
    time.sleep(1)
    if clear_log or not os.path.exists(log_path):
        open(log_path, "w").close()
    else:
        with open(log_path, "a", encoding="utf-8", errors="ignore") as f:
            f.write("\n\n===== Relaunching Fooocus =====\n")

    print(f"Preset: {preset_name}")
    print(f"VRAM: {total_vram_mb} MB -> {vram_flag}")
    print("Launching in background:", " ".join(cmd))
    with open(log_path, "a") as f:
        p = subprocess.Popen(cmd, stdout=f, stderr=subprocess.STDOUT)
    print(f"PID: {p.pid}")
    return p


proc = launch(clear_log=True)
proxy_url = None
link_shown = False
ready = False
restarts = 0
start_time = time.time()

print("Waiting for Fooocus readiness and proxy URL...")
while time.time() - start_time < MAX_WAIT_SECONDS:
    txt = read_log_tail(log_path)
    log_started = ("App started successful" in txt) or ("Running on local URL" in txt)
    ready = local_http_ready(HEALTH_URL)

    if proxy_url is None:
        proxy_url = try_get_proxy_url(PORT)

    # Show link only when backend is actually ready.
    if (not link_shown) and ready and proxy_url:
        print("\n✅ Open this URL:\n" + proxy_url + "\n")
        display(HTML(f'<a href="{proxy_url}" target="_blank">Open Fooocus (Colab proxy)</a>'))
        link_shown = True

    if proc.poll() is not None:
        if restarts < MAX_RESTARTS:
            restarts += 1
            print(f"⚠️ Fooocus exited early with code {proc.returncode}. Auto-retry {restarts}/{MAX_RESTARTS} ...")
            proc = launch(clear_log=False)
            continue
        else:
            break

    if ready and (link_shown or proxy_url):
        break

    if log_started and not ready:
        print("⏳ Server starting (log ready, waiting HTTP health) ...")

    time.sleep(POLL_SECONDS)

if proc.poll() is not None:
    print(f"❌ Fooocus process exited with code {proc.returncode}. Check log below.")
elif ready:
    print("✅ Server is ready.")
else:
    print("⏳ Timeout waiting for server readiness. Try rerunning this launch cell.")

if not link_shown:
    if proxy_url:
        print("\n⚠️ Proxy URL exists but backend not fully ready yet.")
        print("Open it after ~10-20s or rerun this launch cell.")
        print(proxy_url)
    else:
        print("\n⚠️ Could not get Colab proxy URL yet.")
        print("Reconnect runtime tab and rerun this launch cell once.")

print("\nLast log lines:")
subprocess.run(["bash", "-lc", f"tail -n 80 {log_path}"], check=False)


In [ ]:
# Optional: live logs
!tail -n 100 /content/fooocus.log
